# Agentic Fraud Investigation: From Alert to Grounded Recommendation

## 1. Why ML Scoring Is Not Enough

The model answers **“How risky is this transaction?”** The investigation workflow answers **“Why does this alert deserve investigation?”** A human analyst answers **“What action should we actually take?”**

This notebook demonstrates tool selection, evidence retrieval, state, policy grounding, and an explicit workflow without requiring an LLM or paid API.

**How to read the code:** each block corresponds to an explicit tool call or grounding check. The workflow is deterministic so the interview demo runs without an API key.

In [1]:
# Resolve repository-relative paths so the notebook runs from the root or notebooks folder.
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.fraud_tools import FraudTools
from src.investigation_agent import InvestigationAgent

# Instantiate the same small tools used by the production-style workflow.
tools = FraudTools(PROJECT_ROOT)
agent = InvestigationAgent(PROJECT_ROOT)

## 2. Fraud Investigation Architecture

`Alert → transaction lookup → customer history → suspicious signals → policy retrieval → evidence summary → CLOSE / REVIEW / ESCALATE → human decision`

The workflow is deliberately fixed and auditable. An LLM could replace only the final synthesis layer in production.

## 3. Available Agent Tools

1. `get_transaction(transaction_id)` — transaction, model score, and model-visible features.
2. `get_customer_history(customer_id, cutoff)` — history strictly before the alert.
3. `search_fraud_policy(query, top_k=3)` — TF-IDF + cosine similarity over policy sections.
4. `get_risk_score(transaction_id)` — probability, threshold, and display risk level.

## 4. Policy Retrieval

The small synthetic policy is split on Markdown headings. TF-IDF is transparent and reproducible; a production system could replace it with evaluated hybrid/vector retrieval.

In [2]:
# Send only the investigation concepts; the retriever returns the closest policy sections.
display(pd.DataFrame(tools.search_fraud_policy(
    "international high amount new geography rapid transactions new device", top_k=3
))[["section", "similarity", "citation"]])

,section,similarity,citation
0,Account Takeover Indicators,0.413,[Policy: Account Takeover Indicators]
1,New Device,0.204,[Policy: New Device]
2,International Activity,0.176,[Policy: International Activity]


## 5. Investigation Example

Choose a high-scoring test alert with at least three observable signals. The true label is intentionally excluded from the investigation call.

In [3]:
# Load the scored test contract produced by Notebook 1.
scores = pd.read_csv(PROJECT_ROOT / "data" / "sample" / "scored_test_transactions.csv")
# Pick a high-risk alert with multiple visible signals for a concise interview walkthrough.
candidates = scores.loc[(scores.risk_level == "HIGH") & (scores.signal_count >= 3)]
if candidates.empty:
    candidates = scores.loc[scores.risk_level == "HIGH"]
demo_transaction_id = candidates.sort_values("fraud_probability", ascending=False).iloc[0].transaction_id
# Tool 1 returns model-visible evidence but deliberately excludes the true fraud label.
transaction = tools.get_transaction(demo_transaction_id)
display(pd.Series(transaction, name="alert").to_frame())

,alert
transaction_id,TX_035998
customer_id,CUST_03585
transaction_timestamp,2025-11-03T08:28:18
amount,473.16
merchant_category,cash_withdrawal
merchant_country,MA
transaction_channel,online
device_id,DEV_X126152
is_card_present,0
is_international,1


## 6. Evidence Collected

In [4]:
# Tool 2 retrieves customer behavior strictly before the alert timestamp.
history = tools.get_customer_history(transaction["customer_id"], transaction["transaction_timestamp"])
display(pd.Series({k: v for k, v in history.items() if k != "recent_transactions"}, name="history").to_frame())
display(pd.DataFrame(history["recent_transactions"]))

,history
customer_id,CUST_03585
history_cutoff,2025-11-03T08:28:18
transaction_count,13
historical_average_amount,109.69
historical_max_amount,359.28
historical_international_rate,0.0769
known_countries,"[GB, MA]"
known_devices,3
merchant_categories,"[dining, electronics, fuel, grocery, retail, t..."
transactions_previous_1h,0


,transaction_id,transaction_timestamp,amount,merchant_country,merchant_category,transaction_channel,device_id
0,TX_035990,2025-08-17T15:59:09,43.88,MA,fuel,pos,DEV_03585_A
1,TX_035991,2025-09-01T11:47:23,309.13,GB,retail,pos,DEV_03585_A
2,TX_035992,2025-09-03T05:48:11,17.12,GB,grocery,online,DEV_03585_A
3,TX_035993,2025-09-14T21:14:14,30.17,GB,retail,pos,DEV_03585_A
4,TX_035994,2025-09-20T05:35:33,57.79,GB,grocery,pos,DEV_03585_A
5,TX_035995,2025-09-26T21:51:28,14.74,GB,grocery,pos,DEV_03585_A
6,TX_035996,2025-10-23T07:04:19,97.46,GB,fuel,pos,DEV_03585_A
7,TX_035997,2025-10-24T03:08:12,249.04,GB,electronics,online,DEV_X366502


## 7–8. Policy Citations & Agent Recommendation

In [5]:
# Run the fixed tool sequence: transaction → history → policy search → risk score.
result = agent.investigate(demo_transaction_id)
print(json.dumps(result, indent=2))
# Guardrails make the demo auditable: expected tools, real citations, and no unsupported certainty.
assert result["tool_trace"] == ["get_transaction", "get_customer_history", "search_fraud_policy", "get_risk_score"]
assert all(item["citation"].startswith("[Policy: ") for item in result["retrieved_policies"])
assert "definitely fraudulent" not in json.dumps(result).lower()

{
  "transaction_id": "TX_035998",
  "fraud_probability": 0.9005,
  "selected_threshold": 0.6898,
  "risk_level": "HIGH",
  "signals": [
    "Amount $473.16 is 4.3x the customer's prior average.",
    "International activity is unusual relative to the 7.7% prior international rate.",
    "The device was not seen in the customer's prior transactions.",
    "Merchant category 'cash_withdrawal' was not present in prior history."
  ],
  "retrieved_policies": [
    {
      "section": "Merchant-Category Change",
      "citation": "[Policy: Merchant-Category Change]",
      "similarity": 0.342
    },
    {
      "section": "Account Takeover Indicators",
      "citation": "[Policy: Account Takeover Indicators]",
      "similarity": 0.332
    },
    {
      "section": "New Device",
      "citation": "[Policy: New Device]",
      "similarity": 0.226
    }
  ],
  "recommendation": "ESCALATE",
  "reason": "The score exceeds the review threshold and multiple independent signals justify prompt analy

## 9. Human-in-the-Loop

`CLOSE`, `REVIEW`, and `ESCALATE` are recommendations. Only authorized humans may block an account, reverse a payment, freeze funds, contact a customer, or make a regulatory escalation.

## 10. Failure Modes

- **False positives:** legitimate travel, device changes, or large purchases may look unusual.
- **False negatives:** fraud may resemble normal behavior.
- **Retrieval failure:** lexical matching may miss the best policy section.
- **Unsupported claims:** synthesis might say more than the tools returned.
- **Stale policy / model drift:** policy and fraud behavior change.

Mitigations include grounded summaries, citations, human approval, retrieval evaluation, versioned policy, audit logs, and model monitoring.

## 11. Monitoring

- **Model:** PR-AUC, precision, recall, false-positive/negative rates, calibration, score distribution.
- **Operations:** alerts per 1,000, analyst workload, latency, escalation percentage.
- **Agent/RAG:** tool success, retrieval relevance, citation correctness, unsupported-claim rate.
- **Business:** captured fraud, losses, customer friction, false declines, resolution time.

## 12. Key Takeaways

The agent uses tools instead of receiving all context upfront, cites retrieved evidence, and produces a reproducible recommendation. It does not claim certainty and cannot execute high-impact actions.